# 01 - Train Isolation Forest Model

**Elyra Pipeline Node 1** — Train an Isolation Forest anomaly detection model using the training dataset.

**Inputs** (from pipeline or env):
- `TRAIN_DATA_PATH`: Path to training features CSV

**Outputs** (for downstream nodes):
- `MODEL_OUTPUT_PATH`: Saved model pickle
- `FEATURE_COLS_PATH`: JSON with feature column names
- `THRESHOLD_PATH`: Calibrated threshold JSON

In [ ]:
import os
import json
from pathlib import Path

# Elyra/RHOAI: read from pipeline parameters or environment
TRAIN_DATA_PATH = os.getenv("TRAIN_DATA_PATH", "train_data.csv")
MODEL_OUTPUT_PATH = os.getenv("MODEL_OUTPUT_PATH", "artifacts/model.pkl")
FEATURE_COLS_PATH = os.getenv("FEATURE_COLS_PATH", "artifacts/feature_cols.json")
THRESHOLD_PATH = os.getenv("THRESHOLD_PATH", "artifacts/threshold.json")

Path(MODEL_OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd
import numpy as np

# Load training data
if Path(TRAIN_DATA_PATH).exists():
    df = pd.read_csv(TRAIN_DATA_PATH)
    print(f"Loaded {len(df)} rows from {TRAIN_DATA_PATH}")
else:
    # Generate sample training data for demo when no dataset exists
    print("No train data found. Generating sample data...")
    np.random.seed(42)
    n = 500
    feature_cols = ["trace_count", "trace_total_spans", "log_error_count", "log_error_ratio",
                   "metrics_series_len", "req_rate_sum", "trace_http_5xx_spans"]
    X = np.random.exponential(scale=5, size=(n, len(feature_cols))).clip(0, 100)
    df = pd.DataFrame(X, columns=feature_cols)
    df["in_training"] = True
    df["root_cause"] = "none"
    df.to_csv(TRAIN_DATA_PATH, index=False)
    print(f"Saved sample data to {TRAIN_DATA_PATH}")

In [ ]:
# Select feature columns (numeric only)
exclude = {"id", "label", "flag", "variant", "root_cause", "is_baseline", "in_training"}
feature_cols = [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]
train_df = df[df.get("in_training", pd.Series([True] * len(df))) == True]
X_train = train_df[feature_cols].fillna(0).to_numpy()

print(f"Features: {feature_cols}")
print(f"Training samples: {X_train.shape[0]}")

In [ ]:
# Train Isolation Forest
try:
    from isotree import IsolationForest as EIF
    model = EIF(n_estimators=200, random_state=42)
    model.fit(X_train)
    scores_train = model.predict_scores(X_train)
    model_name = "ExtendedIsolationForest(isotree)"
except ImportError:
    from sklearn.ensemble import IsolationForest
    model = IsolationForest(n_estimators=200, contamination="auto", random_state=42)
    model.fit(X_train)
    scores_train = -model.decision_function(X_train)
    model_name = "IsolationForest(sklearn)"

# Calibrate threshold (higher score = more anomalous)
threshold = float(np.quantile(scores_train, 0.995))
print(f"Model: {model_name}")
print(f"Calibrated threshold (0.995 quantile): {threshold:.4f}")

In [ ]:
# Save model and artifacts
from joblib import dump

dump(model, MODEL_OUTPUT_PATH)
Path(FEATURE_COLS_PATH).parent.mkdir(parents=True, exist_ok=True)
with open(FEATURE_COLS_PATH, "w") as f:
    json.dump({"feature_cols": feature_cols, "model_name": model_name}, f, indent=2)
with open(THRESHOLD_PATH, "w") as f:
    json.dump({"threshold": threshold, "quantile": 0.995}, f, indent=2)

print(f"Model saved → {MODEL_OUTPUT_PATH}")
print(f"Feature columns → {FEATURE_COLS_PATH}")
print(f"Threshold → {THRESHOLD_PATH}")